In [1]:
import sys
sys.path.append('../src')
from should_be_stdlib import small_then_big_array
from circuit_postprocess import *
from circuits import (
    circuit_angle_swap,
    circuit_angle_qft_swap,
    circuit_amp_iamp,
    circuit_amp_iamp_qft,
)
from data import *

Operating with: 082620_355l


In [2]:
from itertools import combinations_with_replacement

In [3]:
import pennylane as qml
import pandas as pd
from tqdm.notebook import tqdm

In [4]:
from qbraid import transpile

/home/user/work/quadrigems/.venv/lib/python3.11/site-packages/qbraid/_entrypoints.py:20: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [5]:
tuning_curves_rescaled = pd.read_csv(datapath('data_tuning-curves_rescaled.csv'), index_col=0)
tuning_curves_resampled = pd.read_csv(datapath('data_tuning-curves_resampled.csv'), index_col=0)

adj_tuning_curves = {
    'ang': tuning_curves_rescaled,
    'ang-qft': tuning_curves_rescaled,
    'amp': tuning_curves_resampled,
    'amp-qft': tuning_curves_resampled
}

# Transpile circuits

- To transpile a pennylane circuit to qasm2 (for qiskit), run it under a quantum tape, then use the qbraid transpiler
- IonQ transpiles to a dictionary which can also be saved for later

In [6]:
def circuit_to_tape(pl_circuit):
    def tape_machine(*args):
        with qml.tape.QuantumTape() as tape:
            pl_circuit(*args)
        return tape

    return tape_machine

tape = {
    'ang': circuit_to_tape(circuit_angle_swap),
    'ang-qft': circuit_to_tape(circuit_angle_qft_swap),
    'amp': circuit_to_tape(circuit_amp_iamp),
    'amp-qft': circuit_to_tape(circuit_amp_iamp_qft),
}

In [7]:
def get_fidelity_circuit(name_a_b: tuple[str,int,int]):
    name, a, b = name_a_b
    return [
        a, b,
        transpile(
            tape[name](
                *small_then_big_array(
                    adj_tuning_curves[name].loc[a].to_numpy(),
                    adj_tuning_curves[name].loc[b].to_numpy()
                )
            ),
            'qasm2'
        )
    ]

In [8]:
def get_fidelity_circuits(name):
    datum = adj_tuning_curves[name]
    pairs = combinations_with_replacement(datum.index, 2)
    pairs_len = len(datum) * (len(datum) + 1) // 2

    from multiprocessing import Pool, cpu_count
    with Pool(processes=cpu_count()) as pool:
        ab = list(tqdm(pool.imap(get_fidelity_circuit, [(name,a,b) for (a,b) in pairs]), total=pairs_len))

    return pd.DataFrame(ab, columns=['A', 'B', 'qasm2'])

In [9]:
get_fidelity_circuits('ang').to_excel(datapath('circuits_ang.xlsx'))
get_fidelity_circuits('ang-qft').to_excel(datapath('circuits_ang-qft.xlsx'))
get_fidelity_circuits('amp').to_excel(datapath('circuits_amp.xlsx'))
get_fidelity_circuits('amp-qft').to_excel(datapath('circuits_amp-qft.xlsx'))

  0%|          | 0/283128 [00:00<?, ?it/s]

  0%|          | 0/283128 [00:00<?, ?it/s]

  0%|          | 0/283128 [00:00<?, ?it/s]

  0%|          | 0/283128 [00:00<?, ?it/s]